# UW-Madison GI Tract Image Segmentation  
## Inference and Submission Notebook

This notebook is used only for **test inference and submission generation**.

It does not train the model.

Main workflow:

```text
test images
→ build 2.5D input [slice n-2, slice n, slice n+2]
→ load trained EfficientNet-B5 U-Net checkpoint (aux classification head)
→ predict probability masks (model returns (seg_logits, cls_logits); only seg is used)
→ apply class-specific thresholds
→ resize masks back to original image size
→ C-order RLE encode
→ create submission.csv
```

Important:

```text
This notebook assumes your model was trained with corrected C-order RLE masks.
Therefore, submission RLE encoding also uses C-order flattening.

Model / preprocessing settings (encoder=efficientnet-b5, img_size=456,
aux classification head) must match the training notebook exactly.
```


## 0. Optional Package Installation

Run this only if `segmentation_models_pytorch` is not installed in your Kaggle environment.


In [1]:
import os
import glob
from pathlib import Path

wheel_files = glob.glob("/kaggle/input/**/*.whl", recursive=True)

print(f"Found {len(wheel_files)} wheel files.")

for p in wheel_files[:50]:
    print(p)

if len(wheel_files) == 0:
    raise RuntimeError(
        "No .whl files found under /kaggle/input. "
        "Please add your offline wheels dataset as an input."
    )

wheel_dirs = sorted(set(str(Path(p).parent) for p in wheel_files))

print("\nWheel directories:")
for d in wheel_dirs:
    print(d)

find_links_args = " ".join([f"--find-links={d}" for d in wheel_dirs])

cmd = (
    f"pip install --no-index {find_links_args} "
    f"segmentation-models-pytorch timm"
)

print("\nRunning:")
print(cmd)

os.system(cmd)

import segmentation_models_pytorch as smp
import timm

print("segmentation_models_pytorch version:", smp.__version__)
print("timm version:", timm.__version__)

Found 54 wheel files.
/kaggle/input/datasets/lingxd/uwgi-offline-wheels/wheels/shellingham-1.5.4-py2.py3-none-any.whl
/kaggle/input/datasets/lingxd/uwgi-offline-wheels/wheels/click-8.4.1-py3-none-any.whl
/kaggle/input/datasets/lingxd/uwgi-offline-wheels/wheels/nvidia_cusparselt_cu13-0.8.1-py3-none-manylinux2014_x86_64.whl
/kaggle/input/datasets/lingxd/uwgi-offline-wheels/wheels/typing_extensions-4.15.0-py3-none-any.whl
/kaggle/input/datasets/lingxd/uwgi-offline-wheels/wheels/httpx-0.28.1-py3-none-any.whl
/kaggle/input/datasets/lingxd/uwgi-offline-wheels/wheels/segmentation_models_pytorch-0.5.0-py3-none-any.whl
/kaggle/input/datasets/lingxd/uwgi-offline-wheels/wheels/nvidia_cufft-12.0.0.61-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl
/kaggle/input/datasets/lingxd/uwgi-offline-wheels/wheels/setuptools-81.0.0-py3-none-any.whl
/kaggle/input/datasets/lingxd/uwgi-offline-wheels/wheels/numpy-2.4.6-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl
/kaggle/input/datasets/li

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
libcuml-cu12 26.2.0 requires cuda-toolkit[cublas,cufft,curand,cusolver,cusparse]==12.*, but you have cuda-toolkit 13.0.2 which is incompatible.
cuml-cu12 26.2.0 requires cuda-toolkit[cublas,cufft,curand,cusolver,cusparse]==12.*, but you have cuda-toolkit 13.0.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
torchaudio 2.10.0+cu128 requires torch==2.10.0, but you have torch 2.12.0 which is incompatible.
cudf-cu12 26.2.1 requires cuda-toolkit[nvcc,nvrtc]==12.*, but you have cuda-toolkit 13.0.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cudf-cu12 26.2.1 re

segmentation_models_pytorch version: 0.5.0
timm version: 1.0.26


## 1. Imports and Configuration

In [2]:
import os
import re
import glob
import json
import time
import random
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tqdm.auto import tqdm

import torch
from torch.utils.data import Dataset, DataLoader

import albumentations as A
from albumentations.pytorch import ToTensorV2

try:
    import segmentation_models_pytorch as smp
except ImportError as e:
    raise ImportError(
        "segmentation_models_pytorch is not installed. "
        "Please uncomment and run the installation cell above."
    ) from e

pd.set_option("display.max_columns", 100)

/usr/local/lib/python3.12/dist-packages/albumentations/check_version.py:147: UserWarning: Error fetching version info <urlopen error [Errno -3] Temporary failure in name resolution>
  data = fetch_version_info()


In [ ]:
class CFG:
    seed = 42

    # ------------------------------------------------------------------
    # User-editable paths
    # ------------------------------------------------------------------
    # Competition input folder.
    competition_data_dir = Path("/kaggle/input/competitions/uw-madison-gi-tract-image-segmentation")

    # Fill this with your trained EfficientNet-B5 corrected C-order checkpoint path.
    # Example:
    # checkpoint_path = Path("/kaggle/input/your-b5-dataset/best_effnetb5_unet_c_order_fold0.pth")
    checkpoint_path = Path("/kaggle/input/uwgi-b5/best_effnetb5_unet_c_order_fold0.pth")

    # Optional threshold JSON path.
    # If this file exists, thresholds will be loaded from it.
    # Otherwise, the hard-coded thresholds below will be used.
    # IMPORTANT: the hard-coded thresholds below were tuned for the B0 baseline.
    # For B5, run the threshold-tuning notebook first and point this at the
    # resulting best_thresholds_training_style_c_order_fold0.json.
    thresholds_json_path = Path("/kaggle/input/uwgi-b5-thresholds/best_thresholds_training_style_c_order_fold0.json")

    # ------------------------------------------------------------------
    # Model and preprocessing settings.
    # These must match training.
    # ------------------------------------------------------------------
    classes = ["large_bowel", "small_bowel", "stomach"]

    img_size = 456
    slice_stride = 2

    encoder_name = "efficientnet-b5"
    encoder_weights = None
    in_channels = 3

    # Auxiliary classification head: must match training (the B5 checkpoint was
    # saved with aux_params, so the model returns a (seg_logits, cls_logits) tuple
    # and the checkpoint carries classification_head.* weights).
    use_aux_cls = True

    batch_size = 8
    num_workers = 2

    # Fallback thresholds if no threshold JSON is provided.
    # NOTE: these values were tuned on the B0 baseline; prefer the B5 JSON above.
    thresholds = {
        "large_bowel": 0.39,
        "small_bowel": 0.59,
        "stomach": 0.33,
    }

    # Optional horizontal flip TTA.
    # For the first submission, keep False to validate the pipeline.
    use_tta = False

    # Optional small-mask filtering.
    # For the first submission, keep False.
    use_small_mask_filter = False

    # Areas are measured after resizing prediction back to original image size.
    min_area = {
        "large_bowel": 0,
        "small_bowel": 0,
        "stomach": 0,
    }

    device = "cuda" if torch.cuda.is_available() else "cpu"


def seed_everything(seed=42):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


seed_everything(CFG.seed)

print("=" * 100)
print("Inference configuration")
print("=" * 100)
print("Competition data dir:", CFG.competition_data_dir)
print("Checkpoint path:", CFG.checkpoint_path)
print("Device:", CFG.device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("Image size:", CFG.img_size)
print("Slice stride:", CFG.slice_stride)
print("Encoder:", CFG.encoder_name)
print("Aux classification head:", CFG.use_aux_cls)
print("Use TTA:", CFG.use_tta)
print("Use small-mask filter:", CFG.use_small_mask_filter)
print("Thresholds:", CFG.thresholds)

if CFG.thresholds_json_path is not None and Path(CFG.thresholds_json_path).exists():
    with open(CFG.thresholds_json_path, "r", encoding="utf-8") as f:
        CFG.thresholds = json.load(f)
    print("Loaded thresholds from JSON:", CFG.thresholds_json_path)
    print("Thresholds:", CFG.thresholds)
else:
    print("[Warning] No threshold JSON found; using hard-coded fallback thresholds "
          "(these were tuned for B0, not B5).")

if not CFG.competition_data_dir.exists():
    raise FileNotFoundError(
        f"Competition data directory not found: {CFG.competition_data_dir}. "
        "Please update CFG.competition_data_dir."
    )

if not CFG.checkpoint_path.exists():
    print("\n[Warning] Checkpoint path does not exist.")
    print("Searching for .pth files under /kaggle/input ...")
    possible_paths = glob.glob("/kaggle/input/**/*.pth", recursive=True)
    print(f"Found {len(possible_paths)} .pth files:")
    for p in possible_paths[:30]:
        print("  ", p)
    raise FileNotFoundError(
        f"Checkpoint not found: {CFG.checkpoint_path}\\n"
        "Please update CFG.checkpoint_path using one of the printed paths."
    )
else:
    print("Checkpoint file found.")

## 2. Read Sample Submission

The competition submission file should have columns:

```text
id, class, predicted
```

The final notebook output will overwrite `predicted` with RLE strings.


In [4]:
sample_submission_path = CFG.competition_data_dir / "sample_submission.csv"

if not sample_submission_path.exists():
    raise FileNotFoundError(f"sample_submission.csv not found: {sample_submission_path}")

sample_submission = pd.read_csv(sample_submission_path)

print("Sample submission shape:", sample_submission.shape)
display(sample_submission.head())

required_cols = ["id", "class", "predicted"]
missing_cols = [c for c in required_cols if c not in sample_submission.columns]

if len(missing_cols) > 0:
    raise ValueError(f"sample_submission is missing columns: {missing_cols}")

test_ids_from_submission = sorted(sample_submission["id"].unique().tolist())

print("Number of unique test ids in sample_submission:", len(test_ids_from_submission))
print("Example ids:", test_ids_from_submission[:5])

Sample submission shape: (0, 3)


,id,class,predicted


Number of unique test ids in sample_submission: 0
Example ids: []


## 3. Scan and Parse Test Image Paths

This section scans the test image folder and creates a metadata dataframe.

If the visible public test folder is empty in your Kaggle environment, the notebook will still create an empty `submission.csv` based on `sample_submission.csv`.  
During official submission scoring, Kaggle provides the hidden test images.


In [5]:
def parse_image_path(path):
    """
    Parse metadata from one MRI image path.

    Expected structure:
        test/case123/case123_day20/scans/slice_0001_266_266_1.50_1.50.png
    """
    path = Path(path)
    filename = path.stem

    parts = filename.split("_")
    if len(parts) < 6:
        raise ValueError(f"Unexpected filename format: {filename}")

    slice_id = int(parts[1])
    height = int(parts[2])
    width = int(parts[3])
    spacing_x = float(parts[4])
    spacing_y = float(parts[5])

    case_str = path.parents[2].name
    case_day_str = path.parents[1].name

    case_match = re.search(r"case(\d+)", case_str)
    day_match = re.search(r"case(\d+)_day(\d+)", case_day_str)

    if case_match is None or day_match is None:
        raise ValueError(f"Unexpected folder format: {path}")

    case = int(case_match.group(1))
    day = int(day_match.group(2))

    image_id = f"case{case}_day{day}_slice_{slice_id:04d}"

    return {
        "id": image_id,
        "case": case,
        "day": day,
        "slice": slice_id,
        "height": height,
        "width": width,
        "spacing_x": spacing_x,
        "spacing_y": spacing_y,
        "image_path": str(path),
    }


def build_test_dataframe():
    """
    Build test dataframe by scanning test image paths and adding 2.5D paths.
    """
    test_img_dir = CFG.competition_data_dir / "test"

    if not test_img_dir.exists():
        print(f"[Warning] Test folder not found: {test_img_dir}")
        return pd.DataFrame()

    print("Scanning test image paths ...")
    test_image_paths = sorted(
        glob.glob(str(test_img_dir / "case*" / "case*_day*" / "scans" / "*.png"))
    )

    print("Number of test images found:", len(test_image_paths))

    if len(test_image_paths) == 0:
        return pd.DataFrame()

    print("Parsing test image metadata ...")
    test_meta = pd.DataFrame([parse_image_path(p) for p in test_image_paths])

    # Keep only ids requested by sample_submission if possible.
    if len(test_ids_from_submission) > 0:
        before = len(test_meta)
        test_meta = test_meta[test_meta["id"].isin(test_ids_from_submission)].reset_index(drop=True)
        after = len(test_meta)
        print(f"Filtered test images by sample_submission ids: {before} -> {after}")

    if len(test_meta) == 0:
        print("[Warning] No test metadata remains after filtering.")
        return test_meta

    print("Building 2.5D test paths ...")

    path_dict = {
        (int(row["case"]), int(row["day"]), int(row["slice"])): row["image_path"]
        for _, row in test_meta.iterrows()
    }

    slice_dict = (
        test_meta.groupby(["case", "day"])["slice"]
        .apply(lambda x: sorted(x.astype(int).tolist()))
        .to_dict()
    )

    def get_nearest_slice(available_slices, target_slice):
        available_slices = np.array(available_slices)
        nearest_idx = np.argmin(np.abs(available_slices - target_slice))
        return int(available_slices[nearest_idx])

    prev_paths = []
    center_paths = []
    next_paths = []

    prev_slices = []
    center_slices = []
    next_slices = []

    for _, row in test_meta.iterrows():
        case = int(row["case"])
        day = int(row["day"])
        center_slice = int(row["slice"])
        available_slices = slice_dict[(case, day)]

        target_slices = [
            center_slice - CFG.slice_stride,
            center_slice,
            center_slice + CFG.slice_stride,
        ]

        selected_slices = []
        selected_paths = []

        for s in target_slices:
            nearest_s = get_nearest_slice(available_slices, s)
            selected_slices.append(nearest_s)
            selected_paths.append(path_dict[(case, day, nearest_s)])

        prev_paths.append(selected_paths[0])
        center_paths.append(selected_paths[1])
        next_paths.append(selected_paths[2])

        prev_slices.append(selected_slices[0])
        center_slices.append(selected_slices[1])
        next_slices.append(selected_slices[2])

    test_meta["image_path_prev"] = prev_paths
    test_meta["image_path_center"] = center_paths
    test_meta["image_path_next"] = next_paths

    test_meta["slice_prev"] = prev_slices
    test_meta["slice_center"] = center_slices
    test_meta["slice_next"] = next_slices

    return test_meta


test_df = build_test_dataframe()

print("Test dataframe shape:", test_df.shape)
display(test_df.head())

[Warning] Test folder not found: /kaggle/input/competitions/uw-madison-gi-tract-image-segmentation/test
Test dataframe shape: (0, 0)


""


## 4. Image Reading and Test Dataset

In [6]:
def read_image(path):
    """
    Read a grayscale MRI slice and normalize it to 0-1 using percentile clipping.
    This must match the training preprocessing.
    """
    img = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)

    if img is None:
        raise FileNotFoundError(f"Cannot read image: {path}")

    img = img.astype(np.float32)

    p1 = np.percentile(img, 1)
    p99 = np.percentile(img, 99)

    img = np.clip(img, p1, p99)
    img = (img - p1) / (p99 - p1 + 1e-6)

    return img


def read_2_5d_image(row):
    """
    Read [n-2, n, n+2] slices and stack them as a 3-channel image.
    Neighboring slices are resized to center slice size if needed.
    """
    paths = [
        row["image_path_prev"],
        row["image_path_center"],
        row["image_path_next"],
    ]

    center_img = read_image(row["image_path_center"])
    center_h, center_w = center_img.shape[:2]

    images = []

    for p in paths:
        img = read_image(p)

        if img.shape[:2] != (center_h, center_w):
            img = cv2.resize(
                img,
                (center_w, center_h),
                interpolation=cv2.INTER_LINEAR,
            )

        images.append(img)

    image = np.stack(images, axis=-1)

    return image.astype(np.float32)


def get_test_transforms():
    """
    Test transform. No random augmentation.
    """
    return A.Compose([
        A.Resize(
            CFG.img_size,
            CFG.img_size,
            interpolation=cv2.INTER_LINEAR,
        ),
        ToTensorV2(),
    ])


class UWGITestDataset(Dataset):
    """
    Test dataset for 2.5D inference.
    """
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image = read_2_5d_image(row)

        if self.transform is not None:
            augmented = self.transform(image=image)
            image = augmented["image"]
        else:
            image = cv2.resize(
                image,
                (CFG.img_size, CFG.img_size),
                interpolation=cv2.INTER_LINEAR,
            )
            image = torch.from_numpy(image.transpose(2, 0, 1)).float()

        meta = {
            "id": row["id"],
            "height": int(row["height"]),
            "width": int(row["width"]),
        }

        return image.float(), meta


if len(test_df) > 0:
    test_dataset = UWGITestDataset(test_df, transform=get_test_transforms())

    test_loader = DataLoader(
        test_dataset,
        batch_size=CFG.batch_size,
        shuffle=False,
        num_workers=CFG.num_workers,
        pin_memory=True,
        drop_last=False,
    )

    print("Test dataset length:", len(test_dataset))
    print("Test dataloader batches:", len(test_loader))

    sample_image, sample_meta = test_dataset[0]
    print("Sample image shape:", sample_image.shape, sample_image.dtype)
    print("Sample meta:", sample_meta)
else:
    test_dataset = None
    test_loader = None
    print("No test images found. The notebook will generate an empty-prediction submission.")

No test images found. The notebook will generate an empty-prediction submission.


## 5. Build Model and Load Checkpoint

In [ ]:
def build_model():
    """
    Build the same model architecture used during training.

    When the auxiliary classification head is enabled (B5 experiment), smp.Unet
    is built with aux_params so it returns a (seg_logits, cls_logits) tuple and
    the checkpoint's classification_head.* weights load cleanly under strict=True.
    """
    aux_params = None
    if CFG.use_aux_cls:
        aux_params = dict(
            pooling="avg",
            dropout=0.2,
            classes=len(CFG.classes),
            activation=None,
        )

    model = smp.Unet(
        encoder_name=CFG.encoder_name,
        encoder_weights=CFG.encoder_weights,
        in_channels=CFG.in_channels,
        classes=len(CFG.classes),
        activation=None,
        aux_params=aux_params,
    )
    return model


def forward_seg_logits(model, x):
    """
    Run the model and return only the segmentation logits.

    With the aux head enabled the model returns (seg_logits, cls_logits); we
    discard the classification logits here since the submission only uses masks.
    """
    out = model(x)
    if isinstance(out, (tuple, list)):
        return out[0]
    return out


def load_checkpoint_model(checkpoint_path):
    """
    Load trained model checkpoint.
    """
    print("Building model ...")
    model = build_model()

    print("Loading checkpoint:", checkpoint_path)
    checkpoint = torch.load(checkpoint_path, map_location=CFG.device)

    if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
        state_dict = checkpoint["model_state_dict"]
        print("Checkpoint format: dictionary with model_state_dict")
        print("Checkpoint epoch:", checkpoint.get("epoch", "unknown"))
        print("Best Dice:", checkpoint.get("best_dice", "unknown"))
    else:
        state_dict = checkpoint
        print("Checkpoint format: raw state_dict")

    new_state_dict = {}

    for k, v in state_dict.items():
        if k.startswith("module."):
            new_key = k.replace("module.", "", 1)
        else:
            new_key = k
        new_state_dict[new_key] = v

    model.load_state_dict(new_state_dict, strict=True)
    model = model.to(CFG.device)
    model.eval()

    print("Model loaded successfully.")

    return model


model = load_checkpoint_model(CFG.checkpoint_path)

## 6. C-order RLE Encoding

This project uses corrected C-order RLE logic.

The submission mask is encoded using:

```python
mask.flatten(order="C")
```


In [8]:
def rle_encode(mask):
    """
    Encode a binary mask into an RLE string using C-order flattening.

    Parameters
    ----------
    mask : np.ndarray
        Binary mask with shape (height, width).

    Returns
    -------
    rle : str
        RLE string.
    """
    pixels = mask.astype(np.uint8).flatten(order="C")
    pixels = np.concatenate([[0], pixels, [0]])

    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[0::2]

    return " ".join(str(x) for x in runs)


def apply_small_mask_filter(mask, class_name):
    """
    Remove small predicted masks if enabled.
    """
    if not CFG.use_small_mask_filter:
        return mask

    min_area = CFG.min_area.get(class_name, 0)

    if mask.sum() < min_area:
        return np.zeros_like(mask, dtype=np.uint8)

    return mask

## 7. Run Test Inference

In [ ]:
@torch.no_grad()
def predict_test(model, loader):
    """
    Run test inference and return prediction records.
    """
    model.eval()

    records = []

    threshold_tensor = torch.tensor(
        [CFG.thresholds[c] for c in CFG.classes],
        device=CFG.device,
        dtype=torch.float32,
    ).view(1, len(CFG.classes), 1, 1)

    print("=" * 100)
    print("Start test inference")
    print("=" * 100)
    print("Thresholds:", CFG.thresholds)
    print("Use TTA:", CFG.use_tta)

    for images, metas in tqdm(loader, desc="Inference"):
        images = images.to(CFG.device, non_blocking=True)

        logits = forward_seg_logits(model, images)
        probs = torch.sigmoid(logits)

        if CFG.use_tta:
            images_flip = torch.flip(images, dims=[3])
            logits_flip = forward_seg_logits(model, images_flip)
            probs_flip = torch.sigmoid(logits_flip)
            probs_flip = torch.flip(probs_flip, dims=[3])
            probs = 0.5 * probs + 0.5 * probs_flip

        probs_np = probs.detach().cpu().numpy()

        batch_size = probs_np.shape[0]

        for b in range(batch_size):
            image_id = metas["id"][b]
            original_h = int(metas["height"][b])
            original_w = int(metas["width"][b])

            for class_idx, class_name in enumerate(CFG.classes):
                prob = probs_np[b, class_idx]

                # Resize probability back to original image size.
                prob_resized = cv2.resize(
                    prob,
                    (original_w, original_h),
                    interpolation=cv2.INTER_LINEAR,
                )

                threshold = CFG.thresholds[class_name]
                mask = (prob_resized > threshold).astype(np.uint8)

                mask = apply_small_mask_filter(mask, class_name)

                rle = rle_encode(mask)

                records.append({
                    "id": image_id,
                    "class": class_name,
                    "predicted": rle,
                })

    print("Inference completed.")
    print("Number of prediction records:", len(records))

    return records


if test_loader is not None:
    prediction_records = predict_test(model, test_loader)
    pred_df = pd.DataFrame(prediction_records)
else:
    pred_df = pd.DataFrame(columns=["id", "class", "predicted"])

display(pred_df.head())
print("Prediction dataframe shape:", pred_df.shape)

## 8. Build Submission

The final `submission.csv` must match `sample_submission.csv` order.

If no test images are visible in the current environment, this cell will output empty predictions using `sample_submission.csv`.


In [10]:
def build_submission(sample_submission, pred_df):
    """
    Merge predictions into sample_submission order.
    """
    submission = sample_submission.copy()

    if len(pred_df) == 0:
        print("[Warning] pred_df is empty. Creating empty predictions.")
        submission["predicted"] = ""
        return submission

    pred_df = pred_df.copy()
    pred_df["predicted"] = pred_df["predicted"].fillna("")

    submission = submission.drop(columns=["predicted"]).merge(
        pred_df,
        on=["id", "class"],
        how="left",
    )

    submission["predicted"] = submission["predicted"].fillna("")

    return submission


submission = build_submission(sample_submission, pred_df)

print("Submission shape:", submission.shape)
display(submission.head(10))

print("Number of non-empty predictions:", (submission["predicted"].astype(str).str.len() > 0).sum())
print("Number of empty predictions:", (submission["predicted"].astype(str).str.len() == 0).sum())

submission_path = Path("./submission.csv")
submission.to_csv(submission_path, index=False)

print("Saved submission to:", submission_path)

[Warning] pred_df is empty. Creating empty predictions.
Submission shape: (0, 3)


,id,class,predicted


Number of non-empty predictions: 0
Number of empty predictions: 0
Saved submission to: submission.csv


## 9. Optional Submission Sanity Checks

In [11]:
print("Sample submission columns:", sample_submission.columns.tolist())
print("Submission columns:", submission.columns.tolist())

assert list(submission.columns) == list(sample_submission.columns), "Submission columns do not match sample_submission."
assert len(submission) == len(sample_submission), "Submission length does not match sample_submission."

missing = submission["predicted"].isna().sum()
print("Missing predicted values:", missing)

if missing > 0:
    raise ValueError("Submission contains NaN predictions.")

print("Submission sanity checks passed.")

display(submission.sample(min(10, len(submission)), random_state=CFG.seed))

Sample submission columns: ['id', 'class', 'predicted']
Submission columns: ['id', 'class', 'predicted']
Missing predicted values: 0
Submission sanity checks passed.


,id,class,predicted


## 10. Optional Visualize One Test Prediction

This cell visualizes one test prediction after thresholding.  
It is only for sanity checking and does not affect `submission.csv`.


In [ ]:
@torch.no_grad()
def visualize_one_test_prediction(model, dataset, idx=0):
    """
    Visualize one test image and its predicted masks.
    """
    if dataset is None or len(dataset) == 0:
        print("No test dataset available.")
        return

    image, meta = dataset[idx]
    input_tensor = image.unsqueeze(0).to(CFG.device)

    logits = forward_seg_logits(model, input_tensor)
    probs = torch.sigmoid(logits)

    if CFG.use_tta:
        input_flip = torch.flip(input_tensor, dims=[3])
        logits_flip = forward_seg_logits(model, input_flip)
        probs_flip = torch.sigmoid(logits_flip)
        probs_flip = torch.flip(probs_flip, dims=[3])
        probs = 0.5 * probs + 0.5 * probs_flip

    prob_np = probs[0].detach().cpu().numpy()
    image_np = image.detach().cpu().numpy().transpose(1, 2, 0)
    center_img = image_np[:, :, 1]

    fig, axes = plt.subplots(2, 3, figsize=(15, 9))

    for i, c in enumerate(CFG.classes):
        axes[0, i].imshow(center_img, cmap="gray")
        axes[0, i].imshow(prob_np[i], cmap="jet", alpha=0.45, vmin=0, vmax=1)
        axes[0, i].set_title(f"Probability: {c}, max={prob_np[i].max():.3f}")
        axes[0, i].axis("off")

        pred = (prob_np[i] > CFG.thresholds[c]).astype(np.uint8)

        axes[1, i].imshow(center_img, cmap="gray")
        axes[1, i].imshow(
            np.ma.masked_where(pred == 0, pred),
            cmap="Reds",
            alpha=0.55,
            interpolation="nearest",
        )
        axes[1, i].set_title(f"Prediction: {c}, thr={CFG.thresholds[c]:.2f}")
        axes[1, i].axis("off")

    plt.suptitle(f"Test prediction | id={meta['id']}", fontsize=14)
    plt.tight_layout()
    plt.show()


visualize_one_test_prediction(model, test_dataset, idx=0)

## Final Output

This notebook creates:

```text
submission.csv
```

In Kaggle:

1. Click **Save Version**.
2. Run the notebook.
3. After it finishes, open the output files.
4. Submit `submission.csv` to the competition.

Remember:

```text
You submit submission.csv, not the .pth file.
```
